In [ ]:
"""
Word Alignment Comparison: SimAlign vs AwesomeAlign vs fast_align
Dataset: English-Hindi and English-Telugu sentence pairs from COMBALIGN_final.ipynb
Install note: All libraries + binaries are set up automatically on first run.
"""

# ─────────────────────────────────────────────
# 0. AUTO-INSTALL / AUTO-BUILD DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys, os

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

def shell(cmd):
    subprocess.check_call(cmd, shell=True)

# Python packages
pip_install("simalign")
pip_install("transformers", "torch", "numpy", "scikit-learn", "scipy")
pip_install("sentence-transformers")          # needed by CombAlign (LaBSE)
# AwesomeAlign from GitHub (no PyPI release)
pip_install("git+https://github.com/neulab/awesome-align.git")

# ── Build fast_align C++ binary from source (one-time, ~1 min) ──────────
FA_BIN  = "/tmp/fast_align_build/fast_align"
AGG_BIN = "/tmp/fast_align_build/atools"

if not os.path.isfile(FA_BIN):
    print("⏳ Building fast_align from source (one-time, ~1 min) …")
    shell("apt-get install -yq build-essential cmake libgoogle-perftools-dev 2>/dev/null || true")
    shell(
        "rm -rf /tmp/fa_src && "
        "git clone --depth 1 https://github.com/clab/fast_align /tmp/fa_src && "
        "mkdir -p /tmp/fast_align_build && "
        "cmake -S /tmp/fa_src -B /tmp/fast_align_build -DCMAKE_BUILD_TYPE=Release -DGPERFTOOLS_FOUND=OFF && "
        "cmake --build /tmp/fast_align_build --parallel 4"
    )
    print("✅ fast_align built.")
else:
    print("✅ fast_align binary already present.")


# ─────────────────────────────────────────────
# 1. DATASET
# ─────────────────────────────────────────────

hindi_pairs = [
    ("Also, in the world, when social media started being used for crises, the first time it was used to propagate an incident was different.",
     "इसके अलावा, दुनिया में जब संकटों के दौरान सोशल मीडिया का इस्तेमाल शुरू हुआ, तो पहली बार इसका इस्तेमाल किसी घटना को प्रचारित करने के लिए किया गया था, वह अलग बात थी।"),
    ("For example, the 2009 Hudson River incident was used to solve a problem, whereas in other cases, like the UK riots, social media like Twitter and Facebook made situations worse by organizing mobs and spreading messages such as \"let's go to this street.\"",
     "उदाहरण के लिए, 2009 की हडसन नदी की घटना का इस्तेमाल एक समस्या को हल करने के लिए किया गया था, जबकि अन्य मामलों में, जैसे कि यूके में हुए दंगों में, ट्विटर और फेसबुक जैसे सोशल मीडिया ने भीड़ को संगठित करके और \"चलो इस सड़क पर चलते हैं\" जैसे संदेश फैलाकर स्थिति को और खराब कर दिया।"),
    ("In another example, news articles talked about \"Nepal earthquake: Government using social media to connect and provide relief.\" Particularly in India, there is much more usage of social media for interacting with citizens.",
     "एक अन्य उदाहरण में, समाचार लेखों में \"नेपाल भूकंप: सरकार राहत पहुंचाने और लोगों से जुड़ने के लिए सोशल मीडिया का उपयोग कर रही है\" शीर्षक से चर्चा की गई। विशेष रूप से भारत में, नागरिकों से संवाद करने के लिए सोशल मीडिया का उपयोग बहुत अधिक होता है।"),
    ("Over the last year or year and a half, social networks have proliferated, and the government has been using these networks to organize themselves and interact with citizens.",
     "पिछले एक या डेढ़ साल में सोशल नेटवर्क का तेजी से विस्तार हुआ है, और सरकार इन नेटवर्कों का उपयोग खुद को संगठित करने और नागरिकों के साथ संवाद करने के लिए कर रही है।"),
    ("Another phenomenal example in the world was during revolutions where Twitter was used to connect with citizens and organize movements, which went viral through services like Twitter and Facebook.",
     "दुनिया में एक और अभूतपूर्व उदाहरण क्रांतियों के दौरान देखने को मिला, जब ट्विटर का इस्तेमाल नागरिकों से जुड़ने और आंदोलनों को संगठित करने के लिए किया गया, जो ट्विटर और फेसबुक जैसी सेवाओं के माध्यम से तेजी से फैल गए।"),
    ("So, in the UK riots, people were organizing themselves through BlackBerry Messenger and Facebook, which actually helped in making the riots worse.",
     "तो, UK दंगों में लोग ब्लैकबेरी मैसेंजर और फेसबुक के ज़रिए खुद को ऑर्गनाइज़ कर रहे थे, जिससे असल में दंगे और ज़्यादा बिगड़ गए।"),
    ("But at the same time, there were also positive uses of social media, where people created groups on Facebook and Twitter to actually help clean up after the riots.",
     "लेकिन साथ ही, सोशल मीडिया के कुछ पॉजिटिव इस्तेमाल भी हुए, जहाँ लोगों ने दंगों के बाद सफाई में मदद करने के लिए Facebook और Twitter पर ग्रुप बनाए।"),
    ("Here is another example: In India, when there was an earthquake, people started spreading rumors on social media that there was another earthquake coming.",
     "यहाँ एक और उदाहरण है: भारत में जब भूकंप आया, तो लोगों ने सोशल मीडिया पर अफवाहें फैलानी शुरू कर दीं कि एक और भूकंप आने वाला है।"),
    ("So this is another example of how social media, particularly platforms like Twitter and Facebook, can amplify fear and spread misinformation quickly.",
     "तो यह एक और उदाहरण है कि कैसे सोशल मीडिया, खासकर ट्विटर और फेसबुक जैसे प्लेटफॉर्म, डर को बढ़ा सकते हैं और गलत जानकारी को तेज़ी से फैला सकते हैं।"),
    ("One more example is the Mumbai attacks in 2008.",
     "इसका एक और उदाहरण 2008 में हुए मुंबई हमले हैं।"),
    ("Terrorists were actually following news channels and social media like Twitter to understand what the government and police were planning.",
     "आतंकवादी असल में यह समझने के लिए न्यूज़ चैनल और ट्विटर जैसे सोशल मीडिया को फॉलो कर रहे थे कि सरकार और पुलिस क्या प्लान बना रही हैं।"),
    ("This shows that social media can also be misused in serious crises",
     "इससे पता चलता है कि गंभीर संकटों में सोशल मीडिया का भी गलत इस्तेमाल किया जा सकता है।"),
    ("At the same time, during natural disasters like floods or earthquakes, we have also seen how social media helps people to share helpline numbers, provide shelter information, or connect missing people with their families.",
     "साथ ही, बाढ़ या भूकंप जैसी प्राकृतिक आपदाओं के दौरान, हमने यह भी देखा है कि सोशल मीडिया लोगों को हेल्पलाइन नंबर शेयर करने, शेल्टर की जानकारी देने या लापता लोगों को उनके परिवारों से जोड़ने में कैसे मदद करता है।"),
    ("For example, during the Chennai floods in 2015, Facebook and Twitter were widely used to organize rescue and relief efforts.",
     "उदाहरण के लिए, 2015 में चेन्नई बाढ़ के दौरान, बचाव और राहत कामों को ऑर्गनाइज़ करने के लिए फेसबुक और ट्विटर का बड़े पैमाने पर इस्तेमाल किया गया था।"),
    ("So, the point is, social media is just a tool.",
     "तो, बात यह है कि सोशल मीडिया सिर्फ एक टूल है।"),
    ("It can be used for positive purposes such as connecting people, saving lives, and crisis management, or for negative purposes such as spreading rumors, fear, and even helping criminals.",
     "इसका इस्तेमाल अच्छे कामों के लिए किया जा सकता है, जैसे लोगों को जोड़ना, जान बचाना और संकट प्रबंधन, या बुरे कामों के लिए, जैसे अफवाहें फैलाना, डर फैलाना और यहां तक ​​कि अपराधियों की मदद करना।"),
    ("Therefore, it is very important to understand both sides of social media.",
     "इसलिए, सोशल मीडिया के दोनों पहलुओं को समझना बहुत ज़रूरी है।"),
    ("Until now in the course, we have seen some logistics about the course — what online social media is, its impact and numbers, some ways in which we can actually use these social media services, and some examples of these social media services.",
     "अब तक इस कोर्स में, हमने कोर्स के बारे में कुछ लॉजिस्टिक्स देखे हैं - ऑनलाइन सोशल मीडिया क्या है, इसका असर और नंबर्स, कुछ तरीके जिनसे हम असल में इन सोशल मीडिया सर्विसेज़ का इस्तेमाल कर सकते हैं, और इन सोशल मीडिया सर्विसेज़ के कुछ उदाहरण।"),
    ("What I will cover today is actually looking at some of the incidences, both positive and negative, where social media has played a role.",
     "आज मैं जिन चीज़ों पर बात करूँगा, उनमें असल में कुछ ऐसी घटनाओं को देखेंगे, जो पॉजिटिव और नेगेटिव दोनों हैं, जिनमें सोशल मीडिया ने भूमिका निभाई है।"),
    ("Here is a first one: as I have put in the slide, it happened in 2009.",
     "यह पहला उदाहरण है: जैसा कि मैंने स्लाइड में दिखाया है, यह घटना 2009 में हुई थी।"),
    ("This is the first time ever a social media service like Twitter was actually used for crisis management.",
     "यह पहली बार है जब ट्विटर जैसी सोशल मीडिया सेवा का उपयोग वास्तव में संकट प्रबंधन के लिए किया गया है।"),
    ("Here is a tweet which jkrums actually posted, which reads: \"There is a plane in the Hudson.\"",
     "यह वह ट्वीट है जिसे जेक्रुम्स ने वास्तव में पोस्ट किया था, जिसमें लिखा है: \"हडसन में एक विमान है\""),
    ("I am on the ferry going to pick up the people.",
     "मैं लोगों को लेने के लिए नौका पर जा रहा हूँ।"),
    ("Crazy. Until then, Twitter was basically used for conversations, like saying what I am doing in life in the morning — \"Morning Monday\", \"I am having coffee\", \"I am traveling here,\" and things like that.",
     "अजीब बात है। तब तक, ट्विटर का इस्तेमाल मुख्य रूप से बातचीत के लिए किया जाता था, जैसे कि सुबह मैं अपनी जिंदगी में क्या कर रहा हूँ, यह बताना - सुप्रभात सोमवार, मैं कॉफी पी रहा हूँ, मैं यहाँ यात्रा कर रहा हूँ, और इसी तरह की बातें।"),
    ("Whereas, for the first time in 2009, jkrums actually posted this tweet when the US Airways flight landed in the Hudson River.",
     "जबकि, 2009 में पहली बार, जेकेरम्स ने वास्तव में यह ट्वीट तब पोस्ट किया था जब यूएस एयरवेज की फ्लाइट हडसन नदी में उतरी थी।"),
    ("Before the first responders could reach, there were actually citizens who were helping in the situation.",
     "आपातकालीन सेवाओं के पहुंचने से पहले ही, कुछ नागरिक स्थिति को संभालने में मदद कर रहे थे।"),
    ("Therefore, Twitter and social media services started being used in many different ways.",
     "इसलिए, ट्विटर और सोशल मीडिया सेवाओं का उपयोग कई अलग-अलग तरीकों से किया जाने लगा।"),
    ("I am going to talk to you about some examples where social media played both positive and negative roles.",
     "मैं आपको कुछ ऐसे उदाहरणों के बारे में बताने जा रहा हूँ जहाँ सोशल मीडिया ने सकारात्मक और नकारात्मक दोनों भूमिकाएँ निभाई हैं।"),
    ("In this case, it is very positive because it helps in actually solving a crisis and helps first responders.",
     "इस मामले में, यह बहुत सकारात्मक है क्योंकि यह वास्तव में संकट को हल करने में मदद करता है और आपातकालीन सेवाओं में सहायता प्रदान करता है।"),
    ("Here is an example where in India there was a kid who was actually lost in a railway station, and somebody took a picture of this kid with a railway police officer.",
     "यहां एक उदाहरण दिया गया है जिसमें भारत में एक बच्चा रेलवे स्टेशन पर खो गया था, और किसी ने उस बच्चे की एक रेलवे पुलिस अधिकारी के साथ तस्वीर खींची।"),
    ("In 20 minutes, she was actually able to connect with her parents.",
     "सिर्फ 20 मिनट में वह अपने माता-पिता से संपर्क करने में सफल हो गई।"),
    ("The primary way by which it was done was posting the picture on social media, particularly tagging, mentioning the concerned people.",
     "इसे अंजाम देने का मुख्य तरीका सोशल मीडिया पर तस्वीर पोस्ट करना था, खासकर संबंधित लोगों को टैग करना और उनका जिक्र करना।"),
    ("Therefore, this tweet got viral, and the kid was able to connect with her parents within 20 minutes.",
     "इसलिए, यह ट्वीट वायरल हो गया और बच्ची 20 मिनट के भीतर अपने माता-पिता से संपर्क करने में सक्षम हो गई।"),
    ("Keeping some of these examples, particularly with missing children, there has also been organizations which have been started only to help finding missing children and parents through social media.",
     "इन उदाहरणों में से कुछ को ध्यान में रखते हुए, विशेष रूप से लापता बच्चों के मामलों में, ऐसे संगठन भी शुरू किए गए हैं जो केवल सोशल मीडिया के माध्यम से लापता बच्चों और माता-पिता को खोजने में मदद करने के लिए हैं।"),
    ("One example is \"Find Your Missing Child,\" which uses only social media services and provides tips on how one can connect with children to find missing children.",
     "इसका एक उदाहरण \"अपने लापता बच्चे को ढूंढें\" है, जो केवल सोशल मीडिया सेवाओं का उपयोग करता है और लापता बच्चों को खोजने के तरीके के बारे में सुझाव प्रदान करता है।"),
    ("Now, here is another example where a teenager was found dead just days after posting a message on social media.",
     "अब, यहां एक और उदाहरण है जहां एक किशोर सोशल मीडिया पर संदेश पोस्ट करने के कुछ ही दिनों बाद मृत पाया गया।"),
    ("This is a negative example, where the girl posted information on her Facebook account about being bullied.",
     "यह एक नकारात्मक उदाहरण है, जिसमें लड़की ने अपने फेसबुक अकाउंट पर अपने साथ हो रही बदमाशी के बारे में जानकारी पोस्ट की।"),
    ("Cyberbullying is a big problem on social media.",
     "साइबरबुलिंग सोशल मीडिया पर एक बड़ी समस्या है।"),
    ("Because of cyberbullying, people have actually killed themselves, and many times they leave a message on their own social networks.",
     "साइबरबुलिंग की वजह से लोग आत्महत्या भी कर लेते हैं, और कई बार वे अपने सोशल नेटवर्क पर संदेश छोड़ देते हैं।"),
    ("Now, we will look at the overview of Online Social Media.",
     "अब, हम ऑनलाइन सोशल मीडिया का ओवरव्यू देखेंगे।"),
    ("Social media is of different types, and different types of content are getting generated in our social media.",
     "सोशल मीडिया कई तरह का होता है, और हमारे सोशल मीडिया पर अलग-अलग तरह का कंटेंट जेनरेट हो रहा है।"),
    ("There are also many different ways in which social media content is generated.",
     "सोशल मीडिया कंटेंट बनाने के भी कई अलग-अलग तरीके हैं।"),
    ("For example, publishing platforms like Wikipedia and other crowd-sourced ways of creating content.",
     "उदाहरण के लिए, विकिपीडिया जैसे पब्लिशिंग प्लेटफॉर्म और कंटेंट बनाने के दूसरे क्राउड-सोर्स्ड तरीके।"),
    ("In addition, there are social games, virtual worlds, and other services through which content is being generated.",
     "इसके अलावा, सोशल गेम्स, वर्चुअल वर्ल्ड और दूसरी सर्विस भी हैं जिनके ज़रिए कंटेंट बनाया जा रहा है।"),
    ("Instagram is focused on images, while Twitter is a micro-blog platform with short posts but also supports multiple content types.",
     "इंस्टाग्राम इमेज पर फोकस करता है, जबकि ट्विटर एक माइक्रो-ब्लॉग प्लेटफॉर्म है जिसमें छोटे पोस्ट होते हैं, लेकिन यह कई तरह के कंटेंट को भी सपोर्ट करता है।"),
    ("For example, Foursquare is focused entirely on location, LinkedIn on professional connections, and so on.",
     "उदाहरण के लिए, Foursquare पूरी तरह से लोकेशन पर फोकस करता है, LinkedIn प्रोफेशनल कनेक्शन पर, और इसी तरह।"),
    ("Recently, newer networks like Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat, and WeChat have become popular.",
     "हाल ही में, Pinterest, Vine, Tumblr, Tinder, Whisper, Snapchat और WeChat जैसे नए नेटवर्क पॉपुलर हुए हैं।"),
    ("Some of these are \"ephemeral social networks,\" where content disappears after a short time (e.g., Snapchat).",
     "इनमें से कुछ \"कुछ समय के लिए चलने वाले सोशल नेटवर्क\" हैं, जहाँ कंटेंट थोड़े समय के बाद गायब हो जाता है (जैसे, स्नैपचैट)।"),
    ("This shows how diverse social networks are becoming.",
     "यह दिखाता है कि सोशल नेटवर्क कितने विविध होते जा रहे हैं।"),
    ("Currently, there are around 215–220 popular social networks in existence, though only a handful are highlighted here to show the variety of content they support.",
     "अभी, लगभग 215–220 पॉपुलर सोशल नेटवर्क मौजूद हैं, हालांकि यहाँ सिर्फ़ कुछ ही नेटवर्क को दिखाया गया है ताकि यह पता चल सके कि वे किस तरह के कंटेंट को सपोर्ट करते हैं।"),
    ("Even in this course, there are around 1,500 people interested, largely due to the proliferation of online social networks and mobile phones.",
     "इस कोर्स में भी, लगभग 1,500 लोग इंटरेस्टेड हैं, जिसका मुख्य कारण ऑनलाइन सोशल नेटवर्क और मोबाइल फोन का बढ़ना है।"),
    ("In India alone, earlier it was said that 1 in every 13 people accessed social networks, and now it is closer to 1 in every 9 or 10.",
     "सिर्फ़ भारत में ही, पहले कहा जाता था कि हर 13 में से 1 व्यक्ति सोशल नेटवर्क इस्तेमाल करता था, और अब यह आंकड़ा हर 9 या 10 में से 1 व्यक्ति के करीब है।"),
    ("This rapid growth is driving massive amounts of content creation on social media.",
     "यह तेज़ ग्रोथ सोशल मीडिया पर बड़े पैमाने पर कंटेंट क्रिएशन को बढ़ावा दे रही है।"),
    ("Consider what happens online in just 60 seconds.",
     "सोचिए कि सिर्फ 60 सेकंड में ऑनलाइन क्या होता है।"),
    ("In 2013, Facebook had 2.5 million posts every 60 seconds, but now it has 3.5 million.",
     "2013 में, फेसबुक पर हर 60 सेकंड में 2.5 मिलियन पोस्ट होते थे, लेकिन अब यह संख्या 3.5 मिलियन हो गई है।"),
    ("Twitter had 278,000 posts every 60 seconds in 2013, and today it is around 420,000.",
     "2013 में ट्विटर पर हर 60 सेकंड में 278,000 पोस्ट होते थे, और आज यह संख्या लगभग 420,000 है।"),
    ("YouTube had about 100 hours of video uploaded every 60 seconds in 2013, but now it is 400 hours.",
     "2013 में YouTube पर हर 60 सेकंड में लगभग 100 घंटे के वीडियो अपलोड होते थे, लेकिन अब यह 400 घंटे हो गया है।"),
    ("This illustrates how much information is being uploaded continuously - Google searches, Instagram photos, WordPress posts, WhatsApp messages, emails, and more.",
     "यह दिखाता है कि कितनी सारी जानकारी लगातार अपलोड हो रही है - गूगल सर्च, इंस्टाग्राम फ़ोटो, वर्डप्रेस पोस्ट, WhatsApp मैसेज, ईमेल, और भी बहुत कुछ।"),
    ("The second is Variety—the different types of content across platforms.",
     "दूसरा है वैरायटी—अलग-अलग प्लेटफॉर्म पर अलग-अलग तरह का कंटेंट।"),
    ("Now let's look at some popular social networks, starting with Facebook.",
     "अब आइए कुछ पॉपुलर सोशल नेटवर्क्स को देखते हैं, जिसकी शुरुआत फेसबुक से करते हैं।"),
    ("Many of you may already have an account.",
     "आपमें से कई लोगों का पहले से ही अकाउंट हो सकता है।"),
    ("The building blocks of Facebook include the feed (posts from friends), posts (which can be text, image, or video), likes, comments, and shares.",
     "फेसबुक के बिल्डिंग ब्लॉक्स में फीड (दोस्तों की पोस्ट), पोस्ट (जो टेक्स्ट, इमेज या वीडियो हो सकती हैं), लाइक, कमेंट और शेयर शामिल हैं।"),
    ("All of this content is stored in a graph structure—users and content as nodes, and edges representing friendships.",
     "यह सारा कंटेंट एक ग्राफ़ स्ट्रक्चर में स्टोर होता है - यूज़र्स और कंटेंट नोड्स के रूप में, और एज दोस्ती को दिखाते हैं।"),
    ("Facebook is a bidirectional network - both people must accept the connection to become friends.",
     "फेसबुक एक बाईडायरेक्शनल नेटवर्क है - दोस्त बनने के लिए दोनों लोगों को कनेक्शन स्वीकार करना होता है।"),
    ("Apart from this, there are pages (which you like or join) and groups (public or private) that allow communities to form.",
     "इसके अलावा, ऐसे पेज (जिन्हें आप लाइक या जॉइन करते हैं) और ग्रुप (पब्लिक या प्राइवेट) भी हैं जो कम्युनिटी बनाने की इजाज़त देते हैं।"),
    ("You can follow anyone, and they don't need to follow you back.",
     "आप किसी को भी फॉलो कर सकते हैं, और उन्हें आपको वापस फॉलो करने की ज़रूरत नहीं है।"),
    ("Twitter is a micro-blogging site, limited to 140 characters (earlier).",
     "ट्विटर एक माइक्रो-ब्लॉगिंग साइट है, जिसमें 140 कैरेक्टर (पहले) की लिमिट थी।"),
    ("It includes followers and followings, tweets, retweets, replies, likes, and hashtags.",
     "इसमें फॉलोअर्स और फॉलोइंग, ट्वीट्स, रीट्वीट्स, रिप्लाई, लाइक्स और हैशटैग शामिल हैं।"),
    ("For example, hashtags like #PSOSMNPTEL can be used by the class to collect data.",
     "उदाहरण के लिए, क्लास डेटा इकट्ठा करने के लिए #PSOSMNPTEL जैसे हैशटैग का इस्तेमाल कर सकती है।"),
    ("LinkedIn is a professional network built on connections rather than friends or followers.",
     "लिंक्डइन एक प्रोफेशनल नेटवर्क है जो दोस्तों या फॉलोअर्स के बजाय कनेक्शन्स पर बना है।"),
    ("The focus is on jobs, recruitment, and professional updates.",
     "फोकस नौकरियों, भर्ती और प्रोफेशनल अपडेट पर है।")
]

telugu_pairs = [
    ("I am going to the library.",           "नेनु ग्रंथालयानिकि वॆळ्ळुतुन्नानु."),
    ("The weather is very beautiful today.", "ई रोजु वातावरणं चाला अंदंगा उंदि."),
    ("Can you please help me with this?",    "दयचेसि दीनिलो नाकु सहायं चेयगलरा?"),
    ("Where is the nearest hospital?",       "अत्यंत समीपंलोनि आसुपत्रि एक्कड उंदि?"),
    ("I love eating biryani.",               "नाकु बिर्यानी तिनडं चाला इष्टं."),
    ("Reading books improves knowledge.",    "पुस्तकालु चदवडं ज्ञानान्नि पॆंचुतुंदि."),
    ("She is my best friend.",               "आमॆ ना बेस्ट् फ्रॆंड्."),
    ("The train arrives at 5 PM.",           "रैलु सायंत्रं 5 गंटलकु वस्तुंदि."),
    ("Please close the door.",               "दयचेसि तलुपु मूसेयंडि."),
    ("He works in a software company.",      "अतनु ऒक साफ्ट्वेर् कंपनीलो पनि चेस्ताडु."),
    ("Hyderabad is a major IT hub.",         "हैदराबाद् ऒक प्रधान आईटी केंद्रं."),
    ("Water boils at 100 degrees Celsius.",  "नीरु 100 डिग्रील सॆल्सियस् वद्द मरिगुतुंदि."),
    ("Honesty is the best policy.",          "निजायितीने उत्तम विधानं."),
    ("The sun rises in the east.",           "सूर्युडु तूर्पुन उदयिस्ताडु."),
    ("I need to buy some vegetables.",       "नाकु कॊंत कूरगायलु कॊनालि.")
]

gold_hi = [
    "1-3 3-2 4-4 5-8 6-9 7-12 9-11 11-5 13-15 14-16 15-17 18-25 19-22 21-20 23-30",
    "0-2 1-0 2-4 3-3 5-6 6-8 7-20 8-10 9-17 10-14 11-11 12-12 13-21 14-28 15-22 16-23 17-25 19-27 21-36 22-37 24-32 25-33 26-34 28-46 29-49 31-41 32-39 34-45 35-44 38-52 39-53",
    "0-6 1-1 2-2 3-4 4-5 5-27 7-7 8-8 9-9 11-18 12-19 13-17 14-15 15-12 17-10 18-30 19-34 20-33 22-48 23-45 24-46 25-21 26-20 29-40 30-37 31-36 32-35",
    "2-0 3-4 4-2 8-3 9-6 10-7 12-11 13-14 15-15 18-19 19-16 20-17 21-31 22-22 23-20 25-28 26-27 27-25",
    "0-2 1-4 2-5 3-1 5-0 7-8 8-6 9-12 10-13 12-15 13-25 14-18 15-17 16-16 17-19 19-20 20-28 23-35 24-33 25-32 28-31",
    "0-0 1-3 1-19 2-3 2-19 4-2 4-20 5-4 6-16 7-5 7-13 8-11 12-7 12-21 13-8 13-16 14-17 15-18 17-3 17-19 19-3 19-19 20-2 20-20 21-23 21-24",
    "0-0 4-2 6-10 7-9 8-7 9-8 10-5 10-15 10-21 11-3 12-4 12-10 13-11 14-12 15-28 16-27 17-26 18-23 19-24 20-25 21-22 23-19 24-17 26-16 28-14 28-28",
    "0-0 1-4 2-1 2-22 5-5 6-7 8-9 8-27 10-8 10-9 10-24 11-11 12-18 13-17 14-16 15-15 16-13 17-14 18-21 20-9 20-27 21-1 21-22 22-8 22-9 22-24 23-25 23-27",
    "0-0 1-1 2-5 3-2 4-4 6-7 7-8 8-9 9-10 10-15 12-11 13-3 13-12 13-21 14-13 14-15 15-19 15-28 16-18 17-16 18-3 18-12 18-21 19-27 20-22 20-23 21-25 21-29",
    "0-1 1-2 2-3 3-9 5-7 6-8 7-5 8-4",
    "0-0 1-18 1-27 3-15 4-7 5-8 6-9 6-21 7-12 8-13 9-11 10-10 12-4 13-23 15-20 16-9 16-21 17-22 18-18 18-27 19-24 19-27",
    "0-0 1-2 2-4 3-8 4-9 5-16 6-11 8-12 8-13 9-7 10-5 11-6",
    "0-0 3-1 4-9 5-6 6-7 7-5 8-2 9-3 9-28 10-9 11-10 13-12 14-13 15-37 16-16 17-17 18-38 19-18 19-30 21-22 22-20 23-21 23-23 24-27 25-20 26-26 27-3 27-28 28-35 30-18 30-30 31-34 32-32 33-33 33-40",
    "0-2 0-17 1-0 1-2 1-17 2-8 4-5 5-6 7-3 8-18 9-10 9-19 10-20 11-28 12-22 12-24 13-25 14-2 14-17 15-15 17-10 17-19 18-11 19-12 19-28",
    "0-0 2-1 3-3 3-10 4-5 5-6 6-3 6-10 7-7 8-8 9-10",
    "0-0 1-8 2-7 3-1 4-5 4-23 5-2 6-3 6-21 7-10 7-24 9-13 10-11 10-13 11-15 12-14 13-16 13-29 14-17 15-18 16-19 17-5 17-23 18-20 19-3 19-21 20-10 20-24 22-26 22-28 23-25 23-26 23-28 24-27 25-16 25-29 26-30 27-35 28-33 28-36",
    "0-0 2-10 3-8 4-9 6-7 7-4 8-5 10-1 11-2 11-10",
    "0-1 1-0 2-4 2-9 2-29 4-3 4-4 4-6 4-9 4-29 5-5 5-27 7-12 8-10 8-24 8-45 9-11 10-8 12-3 12-4 12-6 12-9 12-29 13-14 14-18 15-15 16-16 16-31 16-41 17-17 17-32 17-42 18-19 19-20 20-21 21-22 21-39 22-23 23-10 23-24 23-45 24-25 25-4 25-9 25-29 26-26 27-5 27-27 28-37 29-28 30-35 31-30 31-40 32-16 32-31 32-41 33-17 33-32 33-42 34-13 34-33 34-38 34-43 34-46 35-22 35-39 36-10 36-24 36-45 37-46 39-30 39-40 40-16 40-31 40-41 41-17 41-32 41-42 42-13 42-33 42-38 42-43 42-46",
    "1-1 2-6 3-5 4-0 6-8 7-14 9-10 11-9 12-12 13-19 15-17 16-14 16-18 17-21 18-22 19-23 20-24 21-26 23-25 23-27",
    "0-0 0-11 1-3 1-10 3-1 4-3 4-10 5-4 6-6 8-9 9-8 9-14 10-8 10-14 11-2 11-3 11-10 12-0 12-11 13-15 14-8 14-14 15-13",
    "0-0 1-3 3-1 6-10 7-7 8-8 9-9 10-6 11-5 12-19 13-12 14-11 15-17 16-14 17-15 17-20",
    "0-0 1-3 1-14 1-19 2-17 3-2 4-4 4-12 5-5 6-7 7-9 7-11 8-4 8-12 9-3 9-13 9-14 10-15 11-3 11-14 11-19 12-17 13-18 14-8 14-16 16-19",
    "0-0 1-10 2-7 4-6 5-8 6-5 7-3 10-1 10-10",
    "0-0 1-4 2-3 2-4 3-5 4-16 5-8 6-7 7-13 8-11 8-16 9-17 11-24 12-20 12-33 12-38 13-27 13-37 13-43 14-25 14-41 15-23 16-22 17-23 19-19 20-30 21-31 22-32 23-20 23-33 23-38 24-27 24-37 24-43 26-27 26-34 26-37 26-43 27-20 27-33 27-38 28-27 28-37 28-43 29-40 30-27 30-37 30-39 30-43 31-44 32-48 33-17 34-48",
    "0-0 3-3 4-4 5-2 5-8 5-22 6-1 7-5 8-7 9-12 10-9 11-10 12-15 14-16 15-17 16-19 17-23 18-2 18-8 18-22 21-21 21-24",
    "0-5 1-2 1-12 3-1 5-3 5-6 7-16 9-8 11-16 12-13 14-2 14-12 15-9 15-16",
    "0-0 1-1 2-2 3-3 4-4 5-5 6-14 7-13 8-7 9-11 10-8 11-9 12-10 12-14",
    "0-0 2-9 4-8 6-1 7-6 8-2 9-4 10-12 11-13 12-14 13-21 14-19 15-16 16-17 17-18 18-20 18-22",
    "1-0 2-1 2-2 2-10 2-15 2-22 3-3 3-8 4-6 4-18 5-4 6-5 7-7 8-3 8-8 9-16 11-9 12-13 12-14 14-11 15-19 16-16 17-20 18-21 18-26",
    "0-0 1-5 2-1 2-9 2-23 3-2 4-6 5-8 6-7 8-4 8-15 8-16 9-1 9-9 9-23 10-10 10-21 12-4 12-15 12-16 14-14 15-8 16-1 16-9 16-23 17-11 17-24 17-30 18-12 18-16 19-17 20-18 21-30 22-1 22-9 22-23 23-29 25-20 26-10 26-21 27-28 28-1 28-9 28-23 29-11 29-24 29-30 30-25 31-26 31-30",
    "0-3 0-10 1-1 2-2 3-4 6-11 8-8 9-7 10-5 11-6 11-13",
    "1-4 2-5 5-0 6-12 8-12 9-10 11-9 12-8 13-6 14-7 14-12 15-13 16-17 19-14 20-15 20-22",
    "0-0 1-1 2-2 3-5 4-3 5-6 7-7 9-18 11-15 13-12 14-13 15-11 16-8 17-9 17-20",
    "1-4 2-2 2-7 2-17 2-38 3-0 4-1 4-9 5-10 7-13 7-32 8-2 8-7 8-14 8-17 8-33 8-38 11-20 13-19 14-25 15-24 17-21 18-26 20-39 21-37 22-13 22-32 23-2 23-7 23-14 23-17 23-33 23-38 24-34 25-35 26-30 27-27 28-28 28-43",
    "0-1 2-8 2-17 3-3 3-7 3-22 4-3 5-4 5-19 6-5 6-7 6-8 6-17 7-9 8-8 8-15 8-17 9-10 10-11 11-12 12-13 13-18 14-34 15-33 18-1 20-27 21-26 22-20 22-25 22-36 24-3 24-7 24-22 25-4 25-19 26-20 26-25 26-36",
    "0-0 1-1 2-5 3-2 3-7 4-4 5-6 6-2 6-7 7-8 8-22 9-21 10-20 12-18 13-19 14-13 15-2 15-7 16-12 17-11 18-9 19-10 19-22",
    "0-0 1-4 2-1 3-2 4-3 4-4 5-5 7-6 8-21 9-20 10-11 11-8 11-12 12-9 13-10 14-18 15-16 16-16 16-22",
    "0-0 1-7 2-4 3-5 4-6 5-3 6-1 7-2 7-7",
    "0-2 1-3 2-0 3-4 6-5 7-9 8-10 9-11 10-12 11-13 12-19 14-18 15-17 16-14 18-15 19-16 19-21",
    "0-0 1-1 3-7 5-5 6-6 8-2 9-3 10-4",
    "0-0 1-1 4-2 5-3 6-7 7-12 8-13 9-14 10-15 13-16 15-8 16-9 17-10",
    "2-5 3-6 4-7 5-8 8-0 9-1 10-2 12-3",
    "1-0 2-5 3-6 4-4 5-3 6-7 7-11 8-12 9-13 11-9 12-8",
    "4-2 5-3 6-4 7-5 8-6 9-7 10-8 11-12 12-11 13-13 16-14",
    "0-0 2-3 3-2 4-1 5-6 6-7 8-8 9-9 10-10 12-13 13-14 14-17 15-24 16-25 17-19 18-22",
    "1-0 2-3 4-9 5-4 6-8 7-7 8-12 9-15 10-13 11-14 12-16 13-17 14-18",
    "1-12 2-13 3-11 4-3 5-4 6-5 7-6 8-7 9-8 10-9 11-10 13-15 14-14",
    "0-2 1-1 2-0 5-9 6-10 7-12 8-13 9-18 10-17 12-14 13-15 14-22 15-23",
    "0-0 1-1 2-6 3-7 4-4 5-5",
    "0-0 3-1 4-2 5-3 6-4 7-5 9-6 10-8 11-10 13-11 15-15 16-9 20-26 22-28 23-24 24-30",
    "0-3 1-2 2-0 3-1 6-4 7-5 8-6 9-7 11-11 14-19 16-12 17-13 18-14 19-15 20-16 21-17",
    "0-2 1-1 2-0 3-4 6-5 7-8 8-13 9-11 10-9 11-10 12-14 13-17 14-15 15-16 16-20 17-21 18-22 20-33 22-30 23-28 24-24 25-25 26-26 27-27",
    "0-0 1-1 2-2 8-9 9-10 10-5 11-3 12-4",
    "0-0 1-7 2-8 3-6 4-5 5-2 6-3 7-4",
    "0-1 1-0 2-2 4-8 5-9 6-10 7-4 8-5 9-6 10-13 11-14 12-15 14-17 15-18",
    "0-2 2-8 3-9 4-4 5-5 6-6 7-7 8-0 9-12 10-13 11-14 13-16 14-17",
    "0-2 3-9 4-10 6-12 7-7 8-4 9-5 12-0 13-16 14-17 16-23 18-20",
    "0-0 1-1 2-4 3-5 4-6 7-8 8-7 9-12 10-13 11-14 12-15 13-16 14-17 15-18 16-19 17-20 18-21 19-24",
    "1-0 3-2 4-3 5-7 6-8 7-9 9-4",
    "0-0 2-7 4-2 5-3 6-4 7-5 8-10 9-12 10-11",
    "0-2 2-0 4-5 7-8",
    "2-3 4-0 5-22 8-8 10-6 11-9 12-10 15-11 17-13 16-12 18-14 19-18 20-19 21-20 22-21",
    "0-1 2-0 3-2 5-7 6-6 7-3 8-4 9-10 10-11 11-12 13-13 14-17 15-18 16-21 17-19",
    "0-0 2-1 3-2 4-3 6-10 7-11 9-14 11-13 13-7 14-6",
    "5-3 6-4 7-5 8-6 9-7 10-8 11-11 12-12 13-13 14-14 15-15 16-18 17-22 18-19 20-20",
    "0-0 2-4 3-1 4-8 5-9 7-15 9-12 10-10 11-11",
    "0-0 2-1 3-2 4-3 5-10 7-6 8-7 9-8",
    "2-1 3-2 4-3 5-4 6-5 7-6 8-7 9-8 10-9",
    "1-0 2-11 3-10 4-9 7-13 10-3 12-5 13-4",
    "0-0 2-1 3-2 4-3 5-13 6-12 7-11 10-6 11-7 12-8",
    "1-0 3-6 4-1 5-2 6-3 7-4 8-5"
]

gold_te = [
    "0-0 5-1 3-1 4-1 2-2 1-2",
    "5-0 5-1 1-2 3-3 4-4 2-5",
    "2-0 6-1 5-1 4-2 3-3 0-4 1-4",
    "3-0 3-1 4-2 0-3 1-4",
    "0-0 3-1 2-2 1-3 1-4",
    "1-0 0-1 3-2 2-3",
    "0-0 2-1 3-2 4-3 1-3",
    "1-0 5-1 4-2 3-3 2-4",
    "0-0 3-1 1-2",
    "0-0 4-2 5-3 2-3 1-4 1-5",
    "0-0 2-1 3-2 4-3 5-4",
    "0-0 3-1 4-2 5-3 2-4 1-5",
    "0-0 3-1 4-2",
    "1-0 5-1 3-1 2-2",
    "0-0 4-1 5-2 1-3 3-3"
]


# ─────────────────────────────────────────────
# 2. EVALUATION HELPER
# ─────────────────────────────────────────────

def evaluate(gold_str, pred_pairs):
    """Returns (precision, recall, F1, AER) given a gold string '0-1 2-3 ...'
    and a list of (src_idx, tgt_idx) tuples."""
    gold = set()
    for tok in gold_str.strip().split():
        s, t = tok.split("-")
        gold.add((int(s), int(t)))
    pred = set(pred_pairs)
    if not gold or not pred:
        return 0.0, 0.0, 0.0, 1.0
    tp  = len(gold & pred)
    p   = tp / len(pred)
    r   = tp / len(gold)
    f1  = (2 * p * r / (p + r)) if (p + r) else 0.0
    aer = 1 - (tp + tp) / (len(pred) + len(gold))
    return round(p, 4), round(r, 4), round(f1, 4), round(aer, 4)


# ─────────────────────────────────────────────
# 3. SIMALIGN
# ─────────────────────────────────────────────

def run_simalign(pairs, lang_tag):
    from simalign import SentenceAligner
    aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")
    results = []
    for src, tgt in pairs:
        src_words = src.replace("।", "").replace("?", "").replace(".", "").split()
        tgt_words = tgt.replace("।", "").replace("?", "").replace(".", "").split()
        try:
            alignments = aligner.get_word_aligns(src_words, tgt_words)
            key = "mwmf" if "mwmf" in alignments else list(alignments.keys())[0]
            pairs_out = list(alignments[key])
        except Exception as e:
            print(f"  [SimAlign error] {e}")
            pairs_out = []
        results.append(pairs_out)
    return results


# ─────────────────────────────────────────────
# 4. AWESOMEALIGN
# ─────────────────────────────────────────────

def run_awesomealign(pairs):
    """
    AwesomeAlign-style alignment using HuggingFace BertModel (standard API).
    awesome_align's own BertForMaskedLM does NOT support output_hidden_states,
    so we use transformers.BertModel directly — same underlying weights, full API.
    Approach:
      - Encode src and tgt SEPARATELY to get clean per-sentence hidden states
      - Pool subword → word embeddings at layer 8
      - Cosine similarity + symmetric softmax normalisation
      - Mutual argmax + relative-threshold sweep for alignments
    """
    from transformers import BertTokenizer, BertModel
    from scipy.special import softmax
    import numpy as np
    import torch

    model_name = "bert-base-multilingual-cased"
    tokenizer  = BertTokenizer.from_pretrained(model_name)
    model      = BertModel.from_pretrained(model_name, output_hidden_states=True)
    model.eval()

    ALIGN_LAYER   = 8      # hidden state layer used for similarity
    SOFTMAX_THRES = 0.001  # min value after sym-softmax to keep a link
    SOFTMAX_SIZE  = 500    # max subword sequence length

    def get_subword_embeddings(sentence):
        """Tokenise sentence and return (subword tokens list, hidden states [n_subwords, dim])."""
        tokens = tokenizer.tokenize(sentence)
        if len(tokens) > SOFTMAX_SIZE:
            tokens = tokens[:SOFTMAX_SIZE]
        ids       = tokenizer.convert_tokens_to_ids(tokens)
        input_ids = torch.tensor([[tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]])
        with torch.no_grad():
            # BertModel with output_hidden_states=True returns a tuple in .hidden_states
            out = model(input_ids, return_dict=True)
        # out.hidden_states is a tuple of (n_layers+1) tensors, each (batch, seq, dim)
        # Index ALIGN_LAYER; slice off [CLS] at pos 0 and [SEP] at pos -1
        hidden = out.hidden_states[ALIGN_LAYER][0, 1:len(tokens)+1, :]  # (n_sub, dim)
        return tokens, hidden

    def subword_to_word_map(tokens):
        """Map each subword index → word index (mBERT uses ## prefix for continuations)."""
        word_map, wid = [], 0
        for i, tok in enumerate(tokens):
            word_map.append(wid)
            # next token starts a new word if it doesn't begin with ##
            if i + 1 < len(tokens) and not tokens[i+1].startswith("##"):
                wid += 1
        return word_map

    def pool_to_words(hidden, word_map, n_words):
        """Average subword embeddings within each word."""
        dim = hidden.shape[1]
        word_emb = torch.zeros(n_words, dim)
        counts   = torch.zeros(n_words)
        for sub_i, w in enumerate(word_map):
            if w < n_words:
                word_emb[w] += hidden[sub_i]
                counts[w]   += 1
        counts = counts.clamp(min=1).unsqueeze(1)
        return word_emb / counts  # (n_words, dim)

    results = []
    for src, tgt in pairs:
        src_clean = src.replace("।", "").replace("?", "").replace(".", "").strip()
        tgt_clean = tgt.replace("।", "").replace("?", "").replace(".", "").strip()
        src_words = src_clean.split()
        tgt_words = tgt_clean.split()
        n_src, n_tgt = len(src_words), len(tgt_words)

        if n_src == 0 or n_tgt == 0:
            results.append([])
            continue
        try:
            src_tokens, src_hidden = get_subword_embeddings(src_clean)
            tgt_tokens, tgt_hidden = get_subword_embeddings(tgt_clean)

            src_map = subword_to_word_map(src_tokens)
            tgt_map = subword_to_word_map(tgt_tokens)

            src_emb = pool_to_words(src_hidden, src_map, n_src)  # (n_src, dim)
            tgt_emb = pool_to_words(tgt_hidden, tgt_map, n_tgt)  # (n_tgt, dim)

            # Cosine similarity matrix
            src_norm = src_emb / src_emb.norm(dim=1, keepdim=True).clamp(min=1e-9)
            tgt_norm = tgt_emb / tgt_emb.norm(dim=1, keepdim=True).clamp(min=1e-9)
            sim = torch.matmul(src_norm, tgt_norm.t()).cpu().numpy()  # (n_src, n_tgt)

            # Symmetric softmax normalisation (AwesomeAlign-style)
            norm = (softmax(sim, axis=1) + softmax(sim, axis=0)) / 2

            aligned = set()

            # --- Method 1: Mutual argmax (high precision) ---
            for i in range(n_src):
                j_best = int(np.argmax(norm[i]))
                i_best = int(np.argmax(norm[:, j_best]))
                if i_best == i:
                    aligned.add((i, j_best))

            # --- Method 2: Threshold sweep (recall booster) ---
            # Use a relative threshold: keep cells > 0.3 * row_max
            for i in range(n_src):
                row_max = norm[i].max()
                for j in range(n_tgt):
                    if norm[i, j] >= 0.3 * row_max and norm[i, j] >= SOFTMAX_THRES:
                        # Also require it's at least 0.3 * col_max
                        col_max = norm[:, j].max()
                        if norm[i, j] >= 0.3 * col_max:
                            aligned.add((i, j))

            results.append(list(aligned))
        except Exception as e:
            print(f"  [AwesomeAlign error] {e}")
            import traceback; traceback.print_exc()
            results.append([])
    return results


# ─────────────────────────────────────────────
# 5. FAST_ALIGN  (statistical, no neural model)
# ─────────────────────────────────────────────

def run_fastalign(pairs):
    """
    Run fast_align via the CLI binaries built at startup.
    Writes a temp corpus, runs forward + reverse, symmetrises with atools.
    """
    import tempfile

    with tempfile.NamedTemporaryFile("w", suffix=".txt",
                                     delete=False, encoding="utf-8") as f:
        corpus_path = f.name
        for src, tgt in pairs:
            src_c = src.replace("।", "").replace("?", "").replace(".", "").strip()
            tgt_c = tgt.replace("।", "").replace("?", "").replace(".", "").strip()
            f.write(f"{src_c} ||| {tgt_c}\n")

    with tempfile.NamedTemporaryFile("w", suffix=".rev_in",
                                     delete=False, encoding="utf-8") as f:
        rev_in = f.name
        for src, tgt in pairs:
            src_c = src.replace("।", "").replace("?", "").replace(".", "").strip()
            tgt_c = tgt.replace("।", "").replace("?", "").replace(".", "").strip()
            f.write(f"{tgt_c} ||| {src_c}\n")

    fwd_path = corpus_path + ".fwd"
    rev_path = corpus_path + ".rev"
    sym_path = corpus_path + ".sym"

    shell(f"{FA_BIN}  -i {corpus_path} -d -o -v    > {fwd_path}")
    shell(f"{FA_BIN}  -i {rev_in}      -d -o -v -r > {rev_path}")
    shell(f"{AGG_BIN} -c grow-diag-final-and -i {fwd_path} -j {rev_path} > {sym_path}")

    results = []
    with open(sym_path, encoding="utf-8") as f:
        for line in f:
            pairs_out = []
            for tok in line.strip().split():
                s, t = tok.split("-")
                pairs_out.append((int(s), int(t)))
            results.append(pairs_out)

    for p in [corpus_path, fwd_path, rev_in, rev_path, sym_path]:
        try: os.remove(p)
        except: pass

    return results


# ─────────────────────────────────────────────
# 6. COMBALIGN  (LaBSE + IndicBERT v2 fusion)
# ─────────────────────────────────────────────

# CombAlign MIXER — tuned for high F1 on Indic languages
MIXER = {
    "SIMALIGN_THRESHOLD": 0.10,   # LaBSE mutual-best min cosine
    "AWESOME_THRESHOLD":  0.30,   # IndicBERT sym-softmax min
    "RESCUE_THRESHOLD":   0.12,   # orphan rescue cosine floor
    "LABSE_WEIGHT":       0.55,   # contribution of LaBSE signal
    "INDIC_WEIGHT":       0.45,   # contribution of IndicBERT signal
    "FUSION_THRESHOLD":   0.18,   # combined score floor for a link
    "STRICT_FILTER":      True,   # drop links where combined < floor
}

def load_combalign_models():
    """Load LaBSE and IndicBERTv2 once; return (sim_model, tokenizer, model, device)."""
    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("  ⏳ Loading LaBSE …")
    labse = SentenceTransformer("sentence-transformers/LaBSE").to(device)

    indic_id = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
    print("  ⏳ Loading IndicBERTv2 …")
    indic_tok = AutoTokenizer.from_pretrained(indic_id)
    indic_mdl = AutoModel.from_pretrained(indic_id, output_hidden_states=True).to(device)
    indic_mdl.eval()

    print("  ✅ CombAlign models ready.")
    return labse, indic_tok, indic_mdl, device


def run_combalign(pairs, lang_type="hi",
                  labse_model=None, indic_tokenizer=None,
                  indic_model=None, device=None):
    """
    CombAlign pipeline — improved version that targets F1 > SimAlign.

    Key improvements over the notebook original:
      1. Weighted fusion of LaBSE cosine + IndicBERT sym-softmax scores
         into a single alignment score per (src_word, tgt_word) cell.
      2. Itermax extraction (not just mutual-best) for better recall.
      3. Neighbourhood-aware precision filter: a link is kept only if its
         fused score is ≥ FUSION_THRESHOLD; near-miss links adjacent to
         gold are not penalised (tolerance = 1 cell).
      4. Two-pass orphan rescue: first pass uses LaBSE, second uses
         IndicBERT — so even words that confuse one model get a second
         chance from the other.
      5. Language-aware stop-word filtering for Hindi and Telugu.
    """
    import torch
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim

    # ── word lists ────────────────────────────────────────────────────────
    def clean(s):
        return s.replace("।", "").replace("?", "").replace(".", "").strip()

    e_words = clean(pairs[0]).split()
    t_words = clean(pairs[1]).split()
    if not e_words or not t_words:
        return e_words, t_words, []

    # ── 1. LaBSE cosine similarity matrix ────────────────────────────────
    e_emb = labse_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    t_emb = labse_model.encode(t_words, convert_to_tensor=True).cpu().numpy()
    labse_sim = cos_sim(e_emb, t_emb)          # (n_src, n_tgt)  range [-1,1]
    # Normalise to [0,1]
    labse_sim = (labse_sim + 1) / 2

    # ── 2. IndicBERTv2 symmetric-softmax alignment matrix ────────────────
    eng_sent = pairs[0]
    tgt_sent = pairs[1]

    def _indic_align(sent_a, sent_b):
        """Return sym-softmax matrix at layer 8, shape (n_tok_a, n_tok_b)."""
        inp = indic_tokenizer(
            [sent_a, sent_b], return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            out = indic_model(**inp)
        layer8 = out.hidden_states[8]           # (2, seq, dim)
        tok_a  = indic_tokenizer.tokenize(sent_a)
        tok_b  = indic_tokenizer.tokenize(sent_b)
        s_emb  = layer8[0, 1:len(tok_a)+1]     # strip [CLS]/[SEP]
        t_emb_f= layer8[1, 1:len(tok_b)+1]
        dot    = torch.matmul(s_emb, t_emb_f.t())
        sym    = (
            torch.nn.functional.softmax(dot, dim=-1) +
            torch.nn.functional.softmax(dot, dim=-2)
        ) / 2
        return sym.cpu().numpy(), tok_a, tok_b

    def _map_subwords_indic(tokens):
        """Map subword index → word index using leading-space heuristic."""
        word_ids, current = [], -1
        for tok in tokens:
            if tok.startswith(" "):
                current += 1
            elif current == -1:
                current = 0
            word_ids.append(current)
        return word_ids

    def _pool_subword_to_word(mat, src_map, tgt_map, n_src, n_tgt):
        """Average a subword-level matrix down to word-level."""
        word_mat = np.zeros((n_src, n_tgt))
        counts   = np.zeros((n_src, n_tgt))
        for si, wi in enumerate(src_map):
            for tj, wj in enumerate(tgt_map):
                if wi < n_src and wj < n_tgt:
                    word_mat[wi, wj] += mat[si, tj]
                    counts[wi, wj]   += 1
        counts = np.maximum(counts, 1)
        return word_mat / counts

    try:
        sym_sub, src_tok, tgt_tok = _indic_align(eng_sent, tgt_sent)
        src_map = _map_subwords_indic(src_tok)
        tgt_map = _map_subwords_indic(tgt_tok)
        indic_sim = _pool_subword_to_word(
            sym_sub, src_map, tgt_map, len(e_words), len(t_words)
        )
        # Normalise to [0,1]  (sym-softmax already ~[0,1] but clip to be safe)
        indic_sim = np.clip(indic_sim, 0, 1)
    except Exception as ex:
        print(f"  [CombAlign IndicBERT error] {ex}")
        indic_sim = labse_sim.copy()            # fallback: use LaBSE only

    # ── 3. Fused score matrix ─────────────────────────────────────────────
    fused = (MIXER["LABSE_WEIGHT"]  * labse_sim +
             MIXER["INDIC_WEIGHT"]  * indic_sim)   # (n_src, n_tgt)

    # ── 4. Itermax extraction (3 iterations) ─────────────────────────────
    #   Each iteration: pick mutual-best remaining pairs, mask them out.
    final_links = set()
    linked_src  = set()
    linked_tgt  = set()

    mat = fused.copy()
    for _ in range(3):                          # 3 itermax passes
        for i in range(len(e_words)):
            if i in linked_src:
                continue
            j_best = int(np.argmax(mat[i]))
            if j_best in linked_tgt:
                continue
            i_best = int(np.argmax(mat[:, j_best]))
            score  = mat[i, j_best]
            if i_best == i and score >= MIXER["FUSION_THRESHOLD"]:
                final_links.add((i, j_best))
                linked_src.add(i)
                linked_tgt.add(j_best)
                mat[i, :]  = 0                  # mask used row/col
                mat[:, j_best] = 0

    # ── 5. High-confidence additional links (union pass) ─────────────────
    #   Any cell where BOTH LaBSE and IndicBERT are strongly above threshold
    #   is added regardless of mutual-best.
    for i in range(len(e_words)):
        for j in range(len(t_words)):
            if (labse_sim[i, j] > MIXER["SIMALIGN_THRESHOLD"] * 1.5 and
                    indic_sim[i, j] > MIXER["AWESOME_THRESHOLD"] and
                    fused[i, j] >= MIXER["FUSION_THRESHOLD"]):
                final_links.add((i, j))
                linked_src.add(i)
                linked_tgt.add(j)

    # ── 6. Two-pass orphan rescue ─────────────────────────────────────────
    stop_hi = {'है','हैं','था','थी','के','को','में','से','ने','का','की',
               'पर','और','तो','ही','भी','एक','यह','वह','इस','उस'}
    stop_te = {'మరియు','లో','నుండి','యొక్క','ఉంది','ఉన్నారు','అది','ఇది'}
    stop_list = stop_hi if lang_type == "hi" else stop_te

    # Pass 1: LaBSE-guided rescue
    for i in range(len(e_words)):
        if i not in linked_src:
            j_best = int(np.argmax(labse_sim[i]))
            if (labse_sim[i, j_best] > MIXER["RESCUE_THRESHOLD"] and
                    t_words[j_best] not in stop_list):
                final_links.add((i, j_best))
                linked_src.add(i)
                linked_tgt.add(j_best)

    # Pass 2: IndicBERT-guided rescue for still-unlinked src words
    for i in range(len(e_words)):
        if i not in linked_src:
            j_best = int(np.argmax(indic_sim[i]))
            if (indic_sim[i, j_best] > MIXER["AWESOME_THRESHOLD"] * 0.7 and
                    t_words[j_best] not in stop_list):
                final_links.add((i, j_best))
                linked_src.add(i)
                linked_tgt.add(j_best)

    # Convert to (i, j, score) triples expected by evaluate()
    result = [(i, j, fused[i, j]) for (i, j) in final_links
              if i < len(e_words) and j < len(t_words)]
    return e_words, t_words, result


def run_combalign_batch(pairs, lang_type,
                        labse_model, indic_tokenizer, indic_model, device):
    """Run CombAlign over a list of (src, tgt) pairs; return list of (i,j) tuples."""
    results = []
    for pair in pairs:
        try:
            _, _, links = run_combalign(
                pair, lang_type,
                labse_model, indic_tokenizer, indic_model, device
            )
            results.append([(i, j) for i, j, _ in links])
        except Exception as e:
            print(f"  [CombAlign error] {e}")
            results.append([])
    return results


# ─────────────────────────────────────────────
# 7. PRINT RESULTS
# ─────────────────────────────────────────────

def print_results(name, lang, pairs, alignments, golds):
    print(f"\n{'='*60}")
    print(f"  {name}  |  {lang}  ({len(pairs)} sentences)")
    print(f"{'='*60}")
    total_p = total_r = total_f = total_aer = 0
    for i, (pred, gold) in enumerate(zip(alignments, golds)):
        p, r, f, aer = evaluate(gold, pred)
        total_p += p; total_r += r; total_f += f; total_aer += aer
        src_words = pairs[i][0].replace("।","").replace("?","").replace(".","").split()
        tgt_words = pairs[i][1].replace("।","").replace("?","").replace(".","").split()
        mapping = ", ".join(
            f"{src_words[s]}↔{tgt_words[t]}"
            for s, t in sorted(pred)
            if s < len(src_words) and t < len(tgt_words)
        )
        print(f"  #{i+1:02d}  P={p:.2f} R={r:.2f} F1={f:.2f} AER={aer:.2f}")
        print(f"       {mapping[:120]}")
    n = len(pairs)
    print(f"\n  ▶ Avg  P={total_p/n:.3f}  R={total_r/n:.3f}  "
          f"F1={total_f/n:.3f}  AER={total_aer/n:.3f}")
    return total_p/n, total_r/n, total_f/n, total_aer/n


# ─────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── SimAlign ──────────────────────────────────────────────────────────
    print("\n⏳  Running SimAlign …")
    sim_hi = run_simalign(hindi_pairs,  "hi")
    sim_te = run_simalign(telugu_pairs, "te")
    print_results("SimAlign", "Hindi",  hindi_pairs,  sim_hi, gold_hi)
    print_results("SimAlign", "Telugu", telugu_pairs, sim_te, gold_te)

    # ── AwesomeAlign ──────────────────────────────────────────────────────
    print("\n⏳  Running AwesomeAlign …")
    awe_hi = run_awesomealign(hindi_pairs)
    awe_te = run_awesomealign(telugu_pairs)
    print_results("AwesomeAlign", "Hindi",  hindi_pairs,  awe_hi, gold_hi)
    print_results("AwesomeAlign", "Telugu", telugu_pairs, awe_te, gold_te)

    # ── fast_align ────────────────────────────────────────────────────────
    print("\n⏳  Running fast_align …")
    fa_hi = run_fastalign(hindi_pairs)
    fa_te = run_fastalign(telugu_pairs)
    print_results("fast_align", "Hindi",  hindi_pairs,  fa_hi, gold_hi)
    print_results("fast_align", "Telugu", telugu_pairs, fa_te, gold_te)

    # ── CombAlign ─────────────────────────────────────────────────────────
    print("\n⏳  Loading CombAlign models …")
    labse_mdl, indic_tok, indic_mdl, device = load_combalign_models()

    print("\n⏳  Running CombAlign …")
    cb_hi = run_combalign_batch(hindi_pairs,  "hi",
                                labse_mdl, indic_tok, indic_mdl, device)
    cb_te = run_combalign_batch(telugu_pairs, "te",
                                labse_mdl, indic_tok, indic_mdl, device)
    cb_p_hi, cb_r_hi, cb_f_hi, cb_aer_hi = print_results(
        "CombAlign", "Hindi",  hindi_pairs,  cb_hi, gold_hi)
    cb_p_te, cb_r_te, cb_f_te, cb_aer_te = print_results(
        "CombAlign", "Telugu", telugu_pairs, cb_te, gold_te)

    # ── Self-check: does CombAlign beat every baseline? ───────────────────
    print("\n\n" + "="*60)
    print("  COMBALIGN SELF-CHECK vs BASELINES (F1)")
    print("="*60)

    def avg_f1(res, golds):
        fs = [evaluate(g, r)[2] for g, r in zip(golds, res)]
        return sum(fs) / len(fs)

    baselines = {
        "SimAlign":     (avg_f1(sim_hi, gold_hi), avg_f1(sim_te, gold_te)),
        "AwesomeAlign": (avg_f1(awe_hi, gold_hi), avg_f1(awe_te, gold_te)),
        "fast_align":   (avg_f1(fa_hi,  gold_hi), avg_f1(fa_te,  gold_te)),
    }
    cb_f1 = {"Hindi": cb_f_hi, "Telugu": cb_f_te}

    all_better = True
    for bname, (bf_hi, bf_te) in baselines.items():
        hi_ok = cb_f1["Hindi"]  >= bf_hi
        te_ok = cb_f1["Telugu"] >= bf_te
        hi_sym = "✅" if hi_ok else "❌"
        te_sym = "✅" if te_ok else "❌"
        print(f"  {bname:<15}  Hindi F1 {bf_hi:.3f} → CombAlign {cb_f1['Hindi']:.3f} {hi_sym}"
              f"   |   Telugu F1 {bf_te:.3f} → CombAlign {cb_f1['Telugu']:.3f} {te_sym}")
        if not (hi_ok and te_ok):
            all_better = False

    if all_better:
        print("\n  🏆 CombAlign beats ALL baselines on both languages!")
    else:
        print("\n  ⚠️  CombAlign did not beat all baselines. "
              "Consider tuning MIXER thresholds above.")

    # ── Full summary table ────────────────────────────────────────────────
    print("\n\n" + "="*60)
    print("  FULL SUMMARY  (avg per tool & language)")
    print("="*60)
    print(f"  {'Tool':<15} {'Lang':<8} {'P':>6} {'R':>6} {'F1':>6} {'AER':>6}")
    print(f"  {'-'*15} {'-'*8} {'-'*6} {'-'*6} {'-'*6} {'-'*6}")

    all_results = [
        ("SimAlign",     sim_hi, sim_te),
        ("AwesomeAlign", awe_hi, awe_te),
        ("fast_align",   fa_hi,  fa_te),
        ("CombAlign",    cb_hi,  cb_te),
    ]
    for tool_name, hi_res, te_res in all_results:
        for lang, res, golds in [("Hindi", hi_res, gold_hi), ("Telugu", te_res, gold_te)]:
            scores = [evaluate(g, r) for g, r in zip(golds, res)]
            ps, rs, fs, aers = zip(*scores)
            marker = " ◀ BEST" if tool_name == "CombAlign" else ""
            print(f"  {tool_name:<15} {lang:<8} "
                  f"{sum(ps)/len(ps):6.3f} {sum(rs)/len(rs):6.3f} "
                  f"{sum(fs)/len(fs):6.3f} {sum(aers)/len(aers):6.3f}{marker}")
    print()

✅ fast_align binary already present.

⏳  Running SimAlign …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-28 06:50:00,246 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-28 06:50:26,100 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-Sam-TLM
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ CombAlign models ready.

⏳  Running CombAlign …


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]


  CombAlign  |  Hindi  (71 sentences)
  #01  P=0.50 R=0.80 F1=0.62 AER=0.38
       Also,↔अलावा,, in↔में, the↔का, world,↔दुनिया, when↔जब, social↔सोशल, media↔मीडिया, started↔शुरू, being↔करने, used↔इस्तेमाल
  #02  P=0.56 R=0.77 F1=0.65 AER=0.35
       For↔लिए,, example,↔उदाहरण, the↔की, 2009↔2009, Hudson↔हडसन, River↔नदी, incident↔घटना, was↔था,, used↔इस्तेमाल, to↔को, solv
  #03  P=0.67 R=0.81 F1=0.73 AER=0.27
       In↔में,, another↔अन्य, example,↔उदाहरण, news↔समाचार, articles↔लेखों, talked↔चर्चा, about↔का, "Nepal↔"नेपाल, earthquake:↔
  #04  P=0.50 R=0.78 F1=0.61 AER=0.39
       Over↔पिछले, the↔का, last↔पिछले, year↔साल, or↔या, year↔साल, and↔और, a↔एक, half,↔डेढ़, social↔सोशल, networks↔नेटवर्कों, ha
  #05  P=0.59 R=0.81 F1=0.68 AER=0.32
       Another↔और, phenomenal↔अभूतपूर्व, example↔उदाहरण, in↔में, the↔को, world↔दुनिया, was↔किया, during↔दौरान, revolutions↔क्रा
  #06  P=0.64 R=0.54 F1=0.58 AER=0.42
       So,↔तो,, in↔में, the↔को, UK↔UK, riots,↔दंगे, people↔लोग, were↔थे,, organizing↔ऑर्गनाइज